In [1]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [2]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41467-023-36983-2#Sec34"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41467_2023_36983_MOESM4_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name 

## Read in file

In [81]:
excel = pd.read_excel(table_name, skiprows = 0, sheet_name=["14_Jaitin et al."])

df = pd.concat(excel.values())
ctmap = pd.read_csv("ctmap_massier_1.txt", sep="\t", header=None, names=["ctnum", "ctname"], index_col=0)
df["cluster_name"] = df["cluster"].map(ctmap["ctname"])

df

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
0,EEF1A1,2.951128e-187,0.583891,0.998,0.974,4.886478e-183,0,M2 Macrophage
1,S100A6,9.595385e-158,0.676459,0.950,0.829,1.588804e-153,0,M2 Macrophage
2,ADIRF,3.493050e-143,0.632291,0.808,0.581,5.783793e-139,0,M2 Macrophage
3,TPT1,7.105718e-125,0.500452,0.962,0.883,1.176565e-120,0,M2 Macrophage
4,PTMA,4.361358e-116,0.507108,0.919,0.767,7.221537e-112,0,M2 Macrophage
...,...,...,...,...,...,...,...,...
3977,CFD,2.961074e-02,-1.141965,0.613,0.448,1.000000e+00,15,Smooth muscle
3978,ACTG1,4.634163e-02,-0.157431,0.508,0.536,1.000000e+00,15,Smooth muscle
3979,AREG,7.334999e-02,0.174480,0.105,0.075,1.000000e+00,15,Smooth muscle
3980,HLA-A,1.540783e-01,-0.172446,0.387,0.398,1.000000e+00,15,Smooth muscle


In [82]:
df.head()

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
0,EEF1A1,2.951128e-187,0.583891,0.998,0.974,4.886478e-183,0,M2 Macrophage
1,S100A6,9.595385e-158,0.676459,0.950,0.829,1.588804e-153,0,M2 Macrophage
2,ADIRF,3.493050e-143,0.632291,0.808,0.581,5.783793e-139,0,M2 Macrophage
3,TPT1,7.105718e-125,0.500452,0.962,0.883,1.176565e-120,0,M2 Macrophage
4,PTMA,4.361358e-116,0.507108,0.919,0.767,7.221537e-112,0,M2 Macrophage


## Unfiltered

In [83]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster_name": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'M2 Macrophage',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'EEF1A1',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'M2 Macrophage',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'EEF1A1',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'M2 Macrophage',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A6',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'M2 Macrophage',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A6',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_

In [84]:
with open('evidence_unfiltered_jaitin.json', 'w') as f:
    json.dump(result, f, indent = 4)

## Perform the filtering

In [85]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cluster_name" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [86]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [87]:
markers

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
3626,S100A8,0.000000e+00,2.466965,0.706,0.016,0.000000e+00,15,Smooth muscle
3624,S100A9,0.000000e+00,2.443386,0.794,0.021,0.000000e+00,15,Smooth muscle
1917,RNASE1,0.000000e+00,2.460249,0.961,0.166,0.000000e+00,9,M1 Macrophage
169,LUM,0.000000e+00,2.614189,0.962,0.164,0.000000e+00,1,Fibroblast
2517,CLDN5,0.000000e+00,2.648081,0.980,0.095,0.000000e+00,11,Adipocyte
3009,CXCL14,0.000000e+00,2.763135,0.981,0.090,0.000000e+00,13,Fibroblast
3658,FTL,1.621047e-95,2.551337,0.983,0.867,2.684130e-91,15,Smooth muscle
3303,MFAP5,0.000000e+00,2.378398,0.990,0.042,0.000000e+00,14,T cell
170,DCN,0.000000e+00,2.434887,0.991,0.322,0.000000e+00,1,Fibroblast
431,ITLN1,0.000000e+00,2.809476,1.000,0.310,0.000000e+00,2,Endothelial


In [88]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster_name
Smooth muscle    0.827667
M1 Macrophage    0.961000
Fibroblast       0.978000
Adipocyte        0.990000
T cell           0.990000
Endothelial      1.000000
Name: pct.1, dtype: float64

In [89]:
markers[col_ct].value_counts()

cluster_name
Smooth muscle    3
Fibroblast       3
Adipocyte        2
M1 Macrophage    1
T cell           1
Endothelial      1
Name: count, dtype: int64

## Find Ensembl ID


In [12]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [13]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=5):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [14]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [90]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
# df["feature_identifier"] = df["feature_identifier"].apply(run)
df["feature_identifier"] = None
save = df.to_dict(orient = "records")

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'Smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A8',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'Smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A8',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'Smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A9',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'Smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'S100A9',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_ty

## Save evidence

In [91]:
with open("evidence_jaitin.json", "w") as f:
    json.dump(result, f, indent = 4) 